---
## 📡 GNSS-DEMO

Welcome to your hands-on guide for processing GNSS data with SAGE/GAGE! In this notebook, we’ll take you on a two-track journey:

1. **🚶 Traditional Serial Processing**  
   – Download RINEX files one by one to your local GeoLab environment  
   – Run the standard GNSS processing pipeline step-by-step  
   – See how long each stage takes and where the bottlenecks lie  

2. **☁️ Cloud-Optimized Workflow**  
   – Stream data directly from an AWS S3 bucket—no massive local downloads  
   – Parallelize your work with Dask-Gateway for lightning-fast throughput  
   – Compare performance metrics against the serial approach  

Don’t worry if “S3” and “Dask” are unfamiliar terms right now—we’ll introduce each concept with clear, jargon-free explanations right when you need them.

---

## 1️⃣ Traditional Serial Processing
In this section, we’ll explore the classic method of GNSS data handling: downloading RINEX files to your GeoLab environment and processing them one by one.  
RINEX (Receiver Independent Exchange Format) is the open, ASCII-based standard for recording raw GNSS observations—packaging pseudorange, carrier-phase, navigation, and meteorological data into a human-readable, vendor-neutral text format.

**What you’ll do here:**  
1. **Set up storage:** Create a `./rinex_data` directory for your downloads.  
2. **Batch download:** Loop over days 1–9 of 2024 and fetch each RINEX file from the EarthScope archive on S3.  
3. **Parse with GeoRINEX:** Load the GPS L1C signal‐to‐noise ratio (`S1`) into an xarray dataset.  
4. **Compute daily mean SNR:** Calculate the average SNR for each file and record its date.  
5. **Clean up:** Delete each file after processing to keep your workspace tidy.  
6. **Time it all:** Use `time.time()` to capture the total download + processing runtime.  
7. **Visualize results:** Plot daily mean SNR vs. date, overlay the overall average, and annotate the total elapsed time in the title.  

First, we’ll install the GeoRINEX package into a temporary environment using a bash script. For more details, see the [GeoRINEX GitHub repository](https://github.com/geospace-code/georinex/tree/main).

In [ ]:
!pip install georinex

UsageError: Line magic function `%` not found.


In [5]:
# ── Imports & Setup ─────────────────────────────────────────────────────────────

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import georinex as gr                    # GeoRINEX: convert RINEX -> xarray datasets
from datetime import datetime, timedelta

# (Assumes get_es_file is a helper function already defined in your notebook,
#  which downloads a file from a URL into a specified directory.)

# ── Configuration ──────────────────────────────────────────────────────────────

# Directory to store downloaded RINEX files
rinex_dir = "./rinex_data"
os.makedirs(rinex_dir, exist_ok=True)

# Lists to collect timing & SNR results
snr_list   = []    # daily mean signal-to-noise ratios
dates      = []    # corresponding dates
start_time = None  # will hold the timestamp when downloading starts

year = 2024        # the year of observation data

# ── Serial Download & Processing Loop ──────────────────────────────────────────

# Record the start time
start_time = time.time()

# Loop over day-of-year (DOY) 1 through 9
for doy in np.arange(1, 10):
    # 1️⃣ Build the URL for the RINEX file on the EarthScope S3 archive
    filename = f"p057{doy:03d}0.24d.Z"
    url      = f"https://gage-data.earthscope.org/archive/gnss/rinex/obs/{year}/{doy:03d}/{filename}"
    print(f"Downloading: {url}")

    # 2️⃣ Download the file into our rinex_dir
    get_es_file(url, rinex_dir)

    # 3️⃣ Load the downloaded RINEX file into an xarray dataset
    filepath = os.path.join(rinex_dir, filename)
    obs      = gr.load(filepath, use='G', meas=['S1'])  
    #    - use='G' restricts to GPS satellites
    #    - meas=['S1'] loads only the L1C signal-to-noise ratio

    # 4️⃣ Compute the daily average SNR across all epochs & satellites
    daily_mean_snr = obs['S1'].mean().values
    snr_list.append(daily_mean_snr)

    # 5️⃣ Record the corresponding calendar date
    date = datetime(year, 1, 1) + timedelta(days=int(doy - 1))
    dates.append(date)

    # 6️⃣ Clean up: delete the RINEX file to save disk space
    os.remove(filepath)

# Record end time and compute total elapsed time
end_time  = time.time()
total_time = end_time - start_time

# ── Summary Statistics ─────────────────────────────────────────────────────────

overall_mean_snr = np.mean(snr_list)
print(f"Processed {len(snr_list)} days in {total_time:.1f} seconds.")
print(f"Overall mean L1C SNR: {overall_mean_snr:.2f}")

# ── Visualization ──────────────────────────────────────────────────────────────

plt.figure(figsize=(10, 4))
plt.plot(dates, snr_list, marker='o', linestyle='-')
plt.axhline(overall_mean_snr, linestyle='--', label=f"Mean SNR = {overall_mean_snr:.2f}")
plt.title(f"{len(dates)} Days of L1C SNR Averages (Serial Download & Processing)\nTotal time: {total_time:.1f}s")
plt.xlabel("Date")
plt.ylabel("Mean L1C SNR")
plt.xticks(rotation=45)
plt.legend(loc='best')
plt.tight_layout()
plt.show()


Matplotlib is building the font cache; this may take a moment.


Downloading: https://gage-data.earthscope.org/archive/gnss/rinex/obs/2024/001/p0570010.24d.Z


NameError: name 'get_es_file' is not defined

In [4]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 34.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]
